In [1]:
import json
from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.http import models

ROOT = Path.cwd().resolve()
ARTIFACTS_DIR = ROOT / "artifacts"
METADATA_PATH = ARTIFACTS_DIR / "metadata.json"
COLUMN_METADATA_PATH = ARTIFACTS_DIR / "column_metadata.json"
QDRANT_PATH = ARTIFACTS_DIR / "qdrant_storage"
COLLECTION_NAME = "data_ef_datasets"

### Load Metadata

In [2]:
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
column_metadata = json.loads(COLUMN_METADATA_PATH.read_text(encoding='utf-8'))

print(len(metadata), len(column_metadata))

659 659


In [3]:
from dataclasses import dataclass
from typing import Optional, List, Dict, Any

@dataclass
class DatasetRecord:
    id: str
    title: str
    organization_id: Optional[str]
    organization_name: Optional[str]
    category_slug: Optional[str]
    frequency: Optional[str]
    coverage_start: Optional[str]
    coverage_end: Optional[str]
    

In [4]:
records = {}
for item in metadata:
    dataset_id = item.get("id")
    if not dataset_id:
        continue
    organization = item.get("organization") or {}
    categories = item.get("categories") or []
    records[dataset_id] = DatasetRecord(
        id=dataset_id,
        title=item.get("title", ""),
        organization_id=organization.get("id"),
        organization_name=organization.get("name"),
        category_slug=categories[0].get("slug") if categories else None,
        frequency=item.get("frequency"),
        coverage_start=item.get("coverage_start"),
        coverage_end=item.get("coverage_end")
    ) 
print(len(records))

659


### Search by Nearest Neighbors by Dataset ID

In [ ]:
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.http import models

QDRANT_PATH = ARTIFACTS_DIR / "qdrant_storage"
COLLECTION_NAME = "data_ef_datasets"

client = QdrantClient(path=str(QDRANT_PATH))
client.get_collections()



Random dataset ID: pd_689b267e79fe4d000707da79
Dataset record: DatasetRecord(id='pd_689b267e79fe4d000707da79', title='', organization_id=None, organization_name=None, category_slug='public-finance', frequency='once', coverage_start='2002-01-01', coverage_end='2002-12-31')


In [20]:
# get random dataset id
random_state = 203
dataset_ids = list(records.keys())
import random
random.seed(random_state)
random_dataset_id = random.choice(dataset_ids)
print(f"Random dataset ID: {random_dataset_id}")
print(f"Dataset record: {records[random_dataset_id]}")

target = records[random_dataset_id]
results, next_offset = client.scroll(
   collection_name=COLLECTION_NAME,
   scroll_filter=models.Filter(
       must=[
           models.FieldCondition(
               key="dataset_id",
               match=models.MatchValue(value=random_dataset_id)
           )
       ]
    ),
    with_vectors=1,
    limit=1,
)

Random dataset ID: pd_689b267e79fe4d000707da79
Dataset record: DatasetRecord(id='pd_689b267e79fe4d000707da79', title='', organization_id=None, organization_name=None, category_slug='public-finance', frequency='once', coverage_start='2002-01-01', coverage_end='2002-12-31')


Record(id=92, payload={'dataset_id': 'pd_689b267e79fe4d000707da79', 'name': 'National Revenue of Budget Law FY2002', 'category': 'Public Finance', 'org': 'Ministry of Economy and Finance', 'frequency': 'once', 'coverage_start': '2002-01-01', 'coverage_end': '2002-12-31'}, vector=[0.03961578756570816, 0.0668061375617981, -0.004393063485622406, 0.037518374621868134, -0.006408260203897953, -0.017725590616464615, -0.0019803086761385202, -0.003261883044615388, -0.020642481744289398, 0.019354015588760376, -0.035787057131528854, -0.03630414977669716, -0.006467696279287338, -0.016223357990384102, -0.028089333325624466, 0.009535552002489567, 0.01888812705874443, 0.039580680429935455, 0.025176983326673508, 0.08658771961927414, -0.0005069889011792839, 0.009503720328211784, -0.0561765693128109, 0.03533768281340599, -0.02810174971818924, 0.00651765801012516, 0.02468906156718731, 0.008944932371377945, 0.0031769592314958572, -0.028649665415287018, 0.01066872850060463, -0.022740451619029045, -0.081914

In [23]:
target_point = results[0]
target_vector = target_point.vector
target_payload = target_point.payload

nearest_neighbors = client.query_points(
    collection_name=COLLECTION_NAME,
    query=target_vector,
    limit=10
).points

nearest_neighbors

[ScoredPoint(id=92, version=0, score=1.0000000000000004, payload={'dataset_id': 'pd_689b267e79fe4d000707da79', 'name': 'National Revenue of Budget Law FY2002', 'category': 'Public Finance', 'org': 'Ministry of Economy and Finance', 'frequency': 'once', 'coverage_start': '2002-01-01', 'coverage_end': '2002-12-31'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=94, version=0, score=0.9575154292554048, payload={'dataset_id': 'pd_6899ad5079fe4d000707da6b', 'name': 'National Revenue of Budget Law FY2000', 'category': 'Public Finance', 'org': 'Ministry of Economy and Finance', 'frequency': 'once', 'coverage_start': '2000-01-01', 'coverage_end': '2000-12-31'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=93, version=0, score=0.9460352715563435, payload={'dataset_id': 'pd_6899c61b79fe4d000707da6d', 'name': 'National Revenue of Budget Law FY2001', 'category': 'Public Finance', 'org': 'Ministry of Economy and Finance', 'frequency': 'once', 'coverage_start': '